

> Run functional testing (compilation, runtime, test mismatch) on the hybrid experiment results from XMainframe and Qwen


In [ ]:
#HERE
import os
import subprocess
import shutil
import re
import time
from pathlib import Path

# ===== CONFIGURATION =====
TEMP_COMPILE_DIR = '/tmp/temp_compile'
TEST_DATA_DIR = 'CodeNet_Dataset'

os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)

# ===== HELPER FUNCTIONS =====

def extract_main_class_name(java_file_path):
    try:
        with open(java_file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        match = re.search(r'public\s+class\s+(\w+)', content)
        if match:
            return match.group(1)
        match = re.search(r'class\s+(\w+)', content)
        if match:
            return match.group(1)
        return None
    except Exception:
        return None


def compile_java(java_file_path):
    try:
        class_name = extract_main_class_name(java_file_path)
        if not class_name:
            return False, "Could not extract class name", None

        correct_filename = f"{class_name}.java"
        temp_java_path = os.path.join(TEMP_COMPILE_DIR, correct_filename)
        shutil.copy(java_file_path, temp_java_path)

        result = subprocess.run(
            ['javac', temp_java_path],
            capture_output=True, text=True, timeout=10, cwd=TEMP_COMPILE_DIR
        )

        if result.returncode == 0:
            class_file = os.path.join(TEMP_COMPILE_DIR, f'{class_name}.class')
            if os.path.exists(class_file):
                return True, None, class_name
            else:
                return False, "No .class file generated", None
        else:
            return False, result.stderr.strip(), None

    except subprocess.TimeoutExpired:
        return False, "Compilation timeout (>10s)", None
    except Exception as e:
        return False, str(e), None


def run_java_with_input(class_name, input_data):
    try:
        start_time = time.time()
        result = subprocess.run(
            ['java', '-cp', '.', class_name],
            input=input_data, capture_output=True, text=True,
            timeout=5, cwd=TEMP_COMPILE_DIR
        )
        execution_time = time.time() - start_time
        if result.returncode == 0:
            return True, result.stdout.strip(), None, execution_time
        else:
            return False, None, f"Runtime error: {result.stderr.strip()}", None
    except subprocess.TimeoutExpired:
        return False, None, "Execution timeout (>5s)", None
    except Exception as e:
        return False, None, str(e), None


def compare_outputs(actual, expected):
    actual_lines   = [line.strip() for line in actual.strip().split('\n')]
    expected_lines = [line.strip() for line in expected.strip().split('\n')]
    return actual_lines == expected_lines


def clean_temp_dir():
    if os.path.exists(TEMP_COMPILE_DIR):
        shutil.rmtree(TEMP_COMPILE_DIR)
        os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)


# ===== MAIN TEST FUNCTION =====

def test_folder(java_folder, test_data_dir):
    print("=" * 80)
    print("TESTING JAVA FILES (plain Java, no COBOL runtime)")
    print("=" * 80)
    print(f"\n Java folder: {java_folder}")
    print(f" Test data:   {test_data_dir}\n")

    problem_dirs = sorted([
        d for d in os.listdir(java_folder)
        if os.path.isdir(os.path.join(java_folder, d))
    ])

    print(f" Found {len(problem_dirs)} problems\n")

    total              = 0
    passed             = 0
    compilation_failed = 0
    execution_failed   = 0
    output_mismatch    = 0

    all_failures = []  # collect every failure for printing at the end

    for problem_id in problem_dirs:
        problem_java_dir = os.path.join(java_folder, problem_id)
        java_files = [f for f in os.listdir(problem_java_dir) if f.endswith('.java')]

        for java_file in java_files:
            total += 1
            java_path = os.path.join(problem_java_dir, java_file)

            # Compile
            compile_success, compile_error, class_name = compile_java(java_path)
            if not compile_success:
                compilation_failed += 1
                all_failures.append({
                    'type':       'compilation_error',
                    'problem_id': problem_id,
                    'java_file':  java_file,
                    'error':      compile_error,
                })
                clean_temp_dir()
                continue

            # Get test data
            test_input_file  = os.path.join(test_data_dir, problem_id, 'input.txt')
            test_output_file = os.path.join(test_data_dir, problem_id, 'output.txt')

            if not os.path.exists(test_input_file) or not os.path.exists(test_output_file):
                all_failures.append({
                    'type':       'missing_test_data',
                    'problem_id': problem_id,
                    'java_file':  java_file,
                    'error':      'input.txt or output.txt not found',
                })
                clean_temp_dir()
                continue

            with open(test_input_file,  'r', encoding='utf-8') as f:
                test_input = f.read()
            with open(test_output_file, 'r', encoding='utf-8') as f:
                expected_output = f.read()

            # Run
            run_success, actual_output, run_error, exec_time = run_java_with_input(class_name, test_input)
            if not run_success:
                execution_failed += 1
                all_failures.append({
                    'type':       'runtime_error',
                    'problem_id': problem_id,
                    'java_file':  java_file,
                    'error':      run_error,
                })
                clean_temp_dir()
                continue

            # Compare
            if compare_outputs(actual_output, expected_output):
                passed += 1
            else:
                output_mismatch += 1
                all_failures.append({
                    'type':       'output_mismatch',
                    'problem_id': problem_id,
                    'java_file':  java_file,
                    'expected':   expected_output.strip(),
                    'actual':     actual_output.strip(),
                })

            clean_temp_dir()

    # ── Print all failures ──────────────────────────────────────────────────
    if all_failures:
        print("\n" + "=" * 80)
        print(f"ALL FAILURES — {len(all_failures)} total")
        print("=" * 80)
        for item in all_failures:
            print(f"\n  {item['problem_id']}/{item['java_file']}  [{item['type']}]")
            if item['type'] == 'output_mismatch':
                print(f"     Expected: {item['expected'][:300]}")
                print(f"     Actual:   {item['actual'][:300]}")
            else:
                print(f"     Error:    {item['error'][:300]}")

    # ── Summary ─────────────────────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"\n Total: {total}")
    if total > 0:
        print(f" Passed:             {passed}             ({passed/total*100:.1f}%)")
        print(f" Compilation failed: {compilation_failed} ({compilation_failed/total*100:.1f}%)")
        print(f" Execution failed:   {execution_failed}   ({execution_failed/total*100:.1f}%)")
        print(f" Output mismatch:    {output_mismatch}    ({output_mismatch/total*100:.1f}%)")
    else:
        print("  No files found to test!")
    print(f"\n Testing complete!\n")


# ===== USAGE =====

if __name__ == "__main__":
    test_folder(
        java_folder='Dissertation/Hybrid/Results/TOOL NAME/strategy NUMBER',
        test_data_dir='/content/drive/MyDrive/CodeNet_Dataset'
    )



> Run functional testing (compilation, runtime, test mismatch) on the hybrid experiment results for COBOL4J.



In [ ]:
# SETUP
import os, subprocess

print("="*80)
print("INSTALLING OPENSOURCECOBOL4J")
print("="*80)

# Install dependencies
print("\n Installing dependencies...")
os.system("apt-get update -qq")
os.system("apt-get install -y -qq default-jdk build-essential bison flex gettext texinfo libgmp-dev autoconf")

# Download and install opensourcecobol4j
print("\n Downloading opensourcecobol4j v1.1.16...")
os.system("curl -L -o opensourcecobol4j-v1.1.16.tar.gz https://github.com/opensourcecobol/opensourcecobol4j/archive/refs/tags/v1.1.16.tar.gz")

print("\n Extracting...")
os.system("tar zxf opensourcecobol4j-v1.1.16.tar.gz")

print("\n Configuring and building...")
current_dir = os.getcwd()
os.chdir("opensourcecobol4j-1.1.16")
os.system("./configure --prefix=/usr/")
os.system("make")
os.system("make install")

# Set CLASSPATH
print("\n Setting CLASSPATH...")
os.environ['CLASSPATH'] = '/usr/lib/opensourcecobol4j/libcobj.jar'

# Return to original directory
os.chdir(current_dir)

print("\n OpenSourceCOBOL4J installed successfully!")

# Verify installation
result = subprocess.run(['cobj', '--version'], capture_output=True, text=True)
print(f"\n{result.stdout}")


In [ ]:
import os
import subprocess
import shutil
import re
from pathlib import Path

# ===== CONFIGURATION =====
LIBCOBJ_JAR = '/usr/lib/opensourcecobol4j/libcobj.jar'
TEMP_COMPILE_DIR = 'temp_compile'

os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)

# ===== HELPER FUNCTIONS =====

def extract_main_class_name(java_file_path):
    """Extract the main class name from Java source code"""
    try:
        with open(java_file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        # Look for: public class ClassName
        match = re.search(r'public\s+class\s+(\w+)', content)
        if match:
            return match.group(1)

        # Fallback: look for any class declaration
        match = re.search(r'class\s+(\w+)', content)
        if match:
            return match.group(1)

        return None
    except Exception as e:
        return None


def compile_java(java_file_path):
    """Compile a Java file with OpenSourceCOBOL4J runtime"""
    try:
        # Extract class name
        class_name = extract_main_class_name(java_file_path)
        if not class_name:
            return False, "Could not extract class name", None

        # Copy to temp with correct name
        correct_filename = f"{class_name}.java"
        temp_java_path = os.path.join(TEMP_COMPILE_DIR, correct_filename)
        shutil.copy(java_file_path, temp_java_path)

        # Compile
        result = subprocess.run(
            ['javac', '-cp', LIBCOBJ_JAR, temp_java_path],
            capture_output=True,
            text=True,
            timeout=10,
            cwd=TEMP_COMPILE_DIR
        )

        if result.returncode == 0:
            class_file = os.path.join(TEMP_COMPILE_DIR, f'{class_name}.class')
            if os.path.exists(class_file):
                return True, None, class_name
            else:
                return False, "No .class file generated", None
        else:
            return False, result.stderr.strip(), None

    except subprocess.TimeoutExpired:
        return False, "Compilation timeout (>10s)", None
    except Exception as e:
        return False, str(e), None


def run_java_with_input(class_name, input_data):
    """Run compiled Java class with input"""
    try:
        import time
        start_time = time.time()

        result = subprocess.run(
            ['java', '-cp', f'.:{LIBCOBJ_JAR}', class_name],
            input=input_data,
            capture_output=True,
            text=True,
            timeout=5,
            cwd=TEMP_COMPILE_DIR
        )

        execution_time = time.time() - start_time

        if result.returncode == 0:
            return True, result.stdout.strip(), None, execution_time
        else:
            return False, None, f"Runtime error: {result.stderr.strip()}", None

    except subprocess.TimeoutExpired:
        return False, None, "Execution timeout (>5s)", None
    except Exception as e:
        return False, None, str(e), None


def compare_outputs(actual, expected):
    """Compare actual vs expected output"""
    actual_lines = [line.strip() for line in actual.strip().split('\n')]
    expected_lines = [line.strip() for line in expected.strip().split('\n')]
    return actual_lines == expected_lines


def clean_temp_dir():
    """Clean up temporary compilation files"""
    if os.path.exists(TEMP_COMPILE_DIR):
        shutil.rmtree(TEMP_COMPILE_DIR)
        os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)


# ===== MAIN TEST FUNCTION =====

def test_folder(java_folder, codenet_dir):
    """
    Test all Java files in a folder

    Args:
        java_folder: Path to folder containing problem_id subdirectories with .java files
        codenet_dir: Path to CodeNet_Final with test data (input.txt/output.txt)

    Example:
        test_folder('/content/strategyone_results/cleaned', '/content/drive/MyDrive/CodeNet_Final')
    """

    print("="*80)
    print("TESTING JAVA FILES")
    print("="*80)
    print(f"\n Java folder: {java_folder}")
    print(f" Test data: {codenet_dir}\n")

    # Verify libcobj.jar exists
    if not os.path.exists(LIBCOBJ_JAR):
        print(f" ERROR: {LIBCOBJ_JAR} not found!")
        print("Please install OpenSourceCOBOL4J first")
        return

    # Get all problem directories
    problem_dirs = sorted([
        d for d in os.listdir(java_folder)
        if os.path.isdir(os.path.join(java_folder, d))
    ])

    print(f" Found {len(problem_dirs)} problems\n")

    # Stats
    total = 0
    passed = 0
    compilation_failed = 0
    execution_failed = 0
    output_mismatch = 0

    # Process each problem
    for problem_id in problem_dirs:
        problem_java_dir = os.path.join(java_folder, problem_id)

        # Get Java files
        java_files = [f for f in os.listdir(problem_java_dir) if f.endswith('.java')]

        for java_file in java_files:
            total += 1
            java_path = os.path.join(problem_java_dir, java_file)

            #print(f"{'='*70}")
            print(f" Testing {problem_id}/{java_file}")
            #print(f"{'='*70}")

            # Step 1: Compile
            #print(" Compiling...")
            compile_success, compile_error, class_name = compile_java(java_path)

            if not compile_success:
                #print(f" COMPILATION FAILED")
                #print(f"   Error: {compile_error}")
                compilation_failed += 1
                clean_temp_dir()
                continue

            #print(f" Compiled successfully (class: {class_name})")

            # Step 2: Get test data
            test_input_file = os.path.join(codenet_dir, problem_id, 'input.txt')
            test_output_file = os.path.join(codenet_dir, problem_id, 'output.txt')

            if not os.path.exists(test_input_file) or not os.path.exists(test_output_file):
                print(f" Test files not found, skipping execution")
                clean_temp_dir()
                continue

            # Read test data
            with open(test_input_file, 'r', encoding='utf-8') as f:
                test_input = f.read()
            with open(test_output_file, 'r', encoding='utf-8') as f:
                expected_output = f.read()

            # Step 3: Run
            #print(" Running with test input...")
            run_success, actual_output, run_error, exec_time = run_java_with_input(class_name, test_input)

            if not run_success:
                #print(f" EXECUTION FAILED")
                #print(f"   Error: {run_error}")
                execution_failed += 1
                clean_temp_dir()
                continue

            print(f" Executed successfully ({exec_time:.4f}s)")

            # Step 4: Compare outputs
            #print("🔍 Comparing outputs...")
            match = compare_outputs(actual_output, expected_output)

            if match:
                print(f" TEST PASSED - Output matches!")
                passed += 1
            else:
                print(f" OUTPUT MISMATCH")
                print(f"   Expected (first 100 chars): {expected_output[:100]}")
                print(f"   Actual (first 100 chars):   {actual_output[:100]}")
                output_mismatch += 1

            clean_temp_dir()
            print()

    # Print summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"\n Total: {total}")
    if total > 0:
        print(f" Passed: {passed} ({passed/total*100:.1f}%)")
        print(f" Compilation failed: {compilation_failed} ({compilation_failed/total*100:.1f}%)")
        print(f" Execution failed: {execution_failed} ({execution_failed/total*100:.1f}%)")
        print(f" Output mismatch: {output_mismatch} ({output_mismatch/total*100:.1f}%)")
    else:
        print("  No files found to test!")
    print(f"\n Testing complete!\n")


# ===== USAGE EXAMPLE =====

if __name__ == "__main__":
    # Example usage - test the cleaned outputs from strategy one
    test_folder(
        java_folder='Dissertation/Hybrid/Results/COBOL4J/strategy NUMBER',
        codenet_dir='CodeNet_Dataset'
    )




> Getting the statistics of how many tests passed per iteration, given a translator's hybrid experiment results.

In [ ]:
# getting per iteration stats
import json
import sys

# ── Config ──────────────────────────────────────────────────
RESULTS_FILE = 'Dissertation/Hybrid/Results/TOOL NAME/STRATEGY NUMBER/test_results.json'  # adjust path 

# ── Load ────────────────────────────────────────────────────
with open(RESULTS_FILE, 'r', encoding='utf-8') as f:
    results = json.load(f)

total     = len(results)
passed    = [r for r in results if r['passed'] == 'Y']
stagnated = [r for r in results if r.get('stagnated', False) and r['passed'] == 'N']
failed    = [r for r in results if not r.get('stagnated', False) and r['passed'] == 'N']

# ── Pass by iteration ────────────────────────────────────────
pass_by_iter = {}
for r in passed:
    it = r.get('passed_on_iter')
    if it:
        pass_by_iter[it] = pass_by_iter.get(it, 0) + 1

# ── Print summary ────────────────────────────────────────────
print(f"\n{'#'*80}")
print(f"#  FINAL RESULTS (reconstructed from {RESULTS_FILE})")
print(f"{'#'*80}")
print(f"#  Passed:    {len(passed)}/{total} ({len(passed)/total*100:.1f}%)")
print(f"#  Stagnated: {len(stagnated)}/{total} ({len(stagnated)/total*100:.1f}%)")
print(f"#  Failed:    {len(failed)}/{total} ({len(failed)/total*100:.1f}%)")
print(f"#")

cumulative = 0
for it in sorted(pass_by_iter.keys()):
    cumulative += pass_by_iter[it]
    print(f"#   Iter {it}: +{pass_by_iter[it]} passed  (cumulative {cumulative}/{total})")

print(f"{'#'*80}\n")

# ── Breakdown of failures ────────────────────────────────────
error_counts = {}
for r in failed:
    et = r.get('error_type', 'unknown')
    error_counts[et] = error_counts.get(et, 0) + 1

if error_counts:
    print("Failed error breakdown:")
    for et, count in sorted(error_counts.items(), key=lambda x: -x[1]):
        print(f"  {et}: {count}")



> Separating the Dev and Holdout (70/30) split for all translations



In [ ]:
#Split for llm error sets

"""
split_dev_holdout.py
====================
Stratified 70/30 split of error JSONs into dev and holdout sets.
Stratifies by error_type so each split has the same proportion of
compilation, runtime, and test_validation errors.

Produces:
  errors_dev_qwen.json       errors_holdout_qwen.json
  errors_dev_xmainframe.json errors_holdout_xmainframe.json

Usage:
    python split_dev_holdout.py
"""

import json
import random
from collections import defaultdict

# ============================================================
# CONFIGURATION
# ============================================================

RANDOM_SEED = 42  # fixed seed for reproducibility
DEV_RATIO   = 0.7  # 70% dev, 30% holdout

INPUT_FILES = {
    'qwen': 'Dissertation/Stratified_Sampling/OriginalTranslations/errors_qwen.json',
    'xmainframe': 'Dissertation/Stratified_Sampling/OriginalTranslations/errors_xmainframe.json',
}

# ============================================================
# SPLIT LOGIC
# ============================================================

def stratified_split(error_data, dev_ratio, seed):
    """
    Split error_data dict into dev and holdout dicts,
    stratified by error_type.
    """
    random.seed(seed)

    # Group PIDs by error_type
    by_type = defaultdict(list)
    for pid, info in error_data.items():
        etype = info.get('error_type', 'unknown')
        by_type[etype].append(pid)

    dev = {}
    holdout = {}

    for etype, pids in sorted(by_type.items()):
        random.shuffle(pids)
        split_idx = round(len(pids) * dev_ratio)
        dev_pids     = pids[:split_idx]
        holdout_pids = pids[split_idx:]

        for pid in dev_pids:
            dev[pid] = error_data[pid]
        for pid in holdout_pids:
            holdout[pid] = error_data[pid]

    return dev, holdout


def print_stats(name, full, dev, holdout):
    """Print split statistics."""
    # Count by error type
    def count_types(d):
        counts = defaultdict(int)
        for info in d.values():
            counts[info.get('error_type', 'unknown')] += 1
        return dict(sorted(counts.items()))

    full_counts    = count_types(full)
    dev_counts     = count_types(dev)
    holdout_counts = count_types(holdout)

    print(f"\n{'='*60}")
    print(f"  {name.upper()}")
    print(f"{'='*60}")
    print(f"  {'Error Type':<20} {'Total':>6} {'Dev':>6} {'Holdout':>8}")
    print(f"  {'-'*46}")
    for etype in full_counts:
        t = full_counts.get(etype, 0)
        d = dev_counts.get(etype, 0)
        h = holdout_counts.get(etype, 0)
        print(f"  {etype:<20} {t:>6} {d:>6} {h:>8}")
    print(f"  {'-'*46}")
    print(f"  {'TOTAL':<20} {len(full):>6} {len(dev):>6} {len(holdout):>8}")


# ============================================================
# MAIN
# ============================================================

def main():
    print("Stratified Dev/Holdout Split")
    print(f"Ratio: {DEV_RATIO*100:.0f}% dev / {(1-DEV_RATIO)*100:.0f}% holdout")
    print(f"Random seed: {RANDOM_SEED}")

    for name, filepath in INPUT_FILES.items():
        # Load
        with open(filepath, 'r', encoding='utf-8') as f:
            error_data = json.load(f)

        # Split
        dev, holdout = stratified_split(error_data, DEV_RATIO, RANDOM_SEED)

        # Save
        dev_path     = f'errors_dev_{name}.json'
        holdout_path = f'errors_holdout_{name}.json'

        with open(dev_path, 'w', encoding='utf-8') as f:
            json.dump(dev, f, indent=2)
        with open(holdout_path, 'w', encoding='utf-8') as f:
            json.dump(holdout, f, indent=2)

        # Print stats
        print_stats(name, error_data, dev, holdout)
        print(f"\n  Saved: {dev_path} ({len(dev)} entries)")
        print(f"  Saved: {holdout_path} ({len(holdout)} entries)")

    print(f"\n Done!\n")


if __name__ == "__main__":
    main()